# XRD Intensity Visualization

This notebook downloads processed XRD intensity curves for a single sample (IGSN) and overlays them on one plot. It uses `xrd_derived` files with `xrd.csv` in their name, which contain 2theta and intensity values.

Steps:
1. Use `/aimdl/partition` to find all IGSNs that have XRD derived data
2. Choose a target IGSN
3. Download its `xrd.csv` files into memory
4. Plot 2theta vs. intensity for all scans on one figure

In [1]:
import io
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    get_item_to_memory,
    paginate_datafiles,
)

## Find samples with XRD data

Use `/aimdl/partition` to get the list of IGSNs that have at least one `xrd_derived` file. This is faster than scanning all items — the partition endpoint returns only the IGSN strings.

In [2]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    partitions = gc.get('aimdl/partition', parameters={'dataType': 'xrd_derived'})

# The response is a dict keyed by IGSN
igsn_list = list(partitions)
print(f'Found {len(igsn_list)} IGSNs with xrd_derived data')
print('All IGSNs:', igsn_list)

Found 545 IGSNs with xrd_derived data
All IGSNs: ['APLMAD00001//2026-04-27T19:33:29+00:00', 'JHACRD00011//2026-04-17T14:13:45+00:00', 'JHACRD00011//2026-04-17T14:19:01+00:00', 'JHACRD00011//2026-04-17T14:23:49+00:00', 'JHACRD00011//2026-04-17T14:27:17+00:00', 'JHACRD00011//2026-04-17T14:30:29+00:00', 'JHACRD00011//2026-04-17T14:33:29+00:00', 'JHACRD00011//2026-04-17T14:36:05+00:00', 'JHACRD00011//2026-04-17T14:39:31+00:00', 'JHACRD00011//2026-04-17T14:43:09+00:00', 'JHACRD00011//2026-04-17T14:45:47+00:00', 'JHACRD00011//2026-04-17T14:49:51+00:00', 'JHACRD00011//2026-04-17T14:53:42+00:00', 'JHACRD00011//2026-04-17T14:56:21+00:00', 'JHACRD00011//2026-04-17T14:59:17+00:00', 'JHACRD00011//2026-04-17T15:02:14+00:00', 'JHACRD00011//2026-04-17T15:05:24+00:00', 'JHACRD00011//2026-04-17T15:08:12+00:00', 'JHACRD00011//2026-04-17T15:11:44+00:00', 'JHACRD00011//2026-04-17T15:14:31+00:00', 'JHACRD00011//2026-04-17T17:15:45+00:00', 'JHACRD00011//2026-04-17T17:19:34+00:00', 'JHACRD00011//2026-04-17T1

## Choose a target IGSN

Set `TARGET_IGSN` to any IGSN from the list above. The default picks the first one.

In [3]:
TARGET_IGSN = igsn_list[0]  # or set a specific IGSN, e.g. 'JHAMAL00016-002'
print('Target IGSN:', TARGET_IGSN)

Target IGSN: APLMAD00001//2026-04-27T19:33:29+00:00


## Find XRD intensity files for this sample

Filter `xrd_derived` items to those whose name contains `xrd.csv` and whose IGSN matches the target. The filter is applied client-side after each page is fetched.

In [4]:
def is_target_xrd_csv(item):
    return (
        'xrd.csv' in item.get('name', '')
        and item.get('meta', {}).get('igsn', '') == TARGET_IGSN
    )

with requests.Session() as session:
    gc = get_girder_client(session=session)
    items = paginate_datafiles(
        gc,
        'xrd_derived',
        worker_fn=lambda item: item,
        item_filter=is_target_xrd_csv,
        extraFields='["meta.igsn"]',
    )

print(f'Found {len(items)} xrd.csv files for {TARGET_IGSN}')
for item in items:
    print(' -', item['name'])

Found 0 xrd.csv files for APLMAD00001//2026-04-27T19:33:29+00:00


## Download and inspect

Stream each CSV into memory using `get_item_to_memory`. The exact column names depend on how the data was processed — we print them so you know what to plot.

In [5]:
scans = {}
with requests.Session() as session:
    gc = get_girder_client(session=session)
    for item in items:
        content = b''.join(get_item_to_memory(gc, item))
        scans[item['name']] = pd.read_csv(io.BytesIO(content))

if not scans:
    raise RuntimeError(f'No xrd.csv files downloaded for {TARGET_IGSN}. Re-run the cell above to check how many items were found.')

first = next(iter(scans.values()))
print('Columns:', list(first.columns))
print(f'Rows per scan: {len(first)}')
first.head()

RuntimeError: No xrd.csv files downloaded for APLMAD00001//2026-04-27T19:33:29+00:00. Re-run the cell above to check how many items were found.

## Plot 2theta vs. intensity

Overlay all scans for this sample on one figure. Adjust `TWO_THETA_COL` and `INTENSITY_COL` if the column names differ from what's shown above.

In [ ]:
# Update these if the actual column names differ
TWO_THETA_COL = first.columns[0]
INTENSITY_COL = first.columns[1]

fig, ax = plt.subplots(figsize=(11, 5))
for name, scan_df in scans.items():
    ax.plot(
        scan_df[TWO_THETA_COL],
        scan_df[INTENSITY_COL],
        label=name,
        alpha=0.8,
        linewidth=1,
    )

ax.set_xlabel(TWO_THETA_COL)
ax.set_ylabel(INTENSITY_COL)
ax.set_title(f'XRD Intensity — {TARGET_IGSN} ({len(scans)} scan(s))')
if len(scans) <= 10:
    ax.legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()